<div style="background:linear-gradient(120deg,#0b4a43 0%,#0e7a6e 55%,#0f6cbd 100%);background-color:#0e7a6e;border-radius:12px;padding:22px 28px;color:#ffffff"><table style="border:none;border-collapse:collapse;width:100%;margin:0"><tr style="border:none;background:none"><td style="border:none;padding:0;vertical-align:middle;text-align:left;color:#ffffff"><div style="font-size:13px;letter-spacing:2px;text-transform:uppercase;opacity:0.85">Microsoft Fabric &middot; Lakehouse</div><div style="font-size:34px;font-weight:700;line-height:1.2;margin:2px 0 6px 0">Restore Utility</div><div style="font-size:15px;line-height:1.5;max-width:520px">Brings a backup back from the write-once ADLS Gen2 archive into a lakehouse: Delta tables, with their history, and Files. The archive itself is never changed.</div></td><td style="border:none;padding:0 0 0 20px;vertical-align:middle;text-align:right"></td></tr></table></div><div style="margin:12px 2px 0 2px;font-size:14px;line-height:1.6"><b>How to use this notebook:</b> run the cells from top to bottom, <b>one at a time</b>. Do not use <i>Run all</i>: the preview and restore must wait until the form is filled in.<br/><span style="display:inline-block;background:#107c10;color:#ffffff;border-radius:11px;padding:1px 10px;font-size:12px;font-weight:600">Steps 1 to 3 change nothing</span> &nbsp; <span style="display:inline-block;background:#c4314b;color:#ffffff;border-radius:11px;padding:1px 10px;font-size:12px;font-weight:600">Step 4 writes to the target lakehouse</span></div><div style="margin:10px 2px 0 2px;font-size:13px;line-height:1.55;opacity:0.9"><b>Before you start:</b> know which backup you want (the timestamp from the backup run) and which lakehouse to restore into. An empty lakehouse is the safest target. Not restored: shortcuts, permissions, security roles, SQL endpoint views.</div>


<div style="border-left:6px solid #5c2e91;border-radius:6px;padding:10px 16px;margin:14px 0 4px 0;background:rgba(127,127,127,0.09)"><table style="border:none;border-collapse:collapse;margin:0"><tr style="border:none;background:none"><td style="border:none;padding:0 14px 0 0;vertical-align:top"></td><td style="border:none;padding:0;vertical-align:top;text-align:left"><div style="font-size:12px;letter-spacing:1.5px;text-transform:uppercase;color:#5c2e91;font-weight:700">Step 1</div><div style="font-size:20px;font-weight:600;margin:1px 0 6px 0">Load the restore functions</div><div style="font-size:14px;line-height:1.55">Loads the functions that do the preview and the restore from <code>nb_backup_restore_adls_v6</code>. It copies nothing.<br/><b>Expect:</b> the cell finishes in a few seconds with no output. <b>If a later step says</b> <code>name ... is not defined</code>, this cell was not run in the current session: run it again.</div></td></tr></table></div>


In [ ]:
%run nb_backup_restore_adls_v6

<div style="border-left:6px solid #0f6cbd;border-radius:6px;padding:10px 16px;margin:14px 0 4px 0;background:rgba(127,127,127,0.09)"><table style="border:none;border-collapse:collapse;margin:0"><tr style="border:none;background:none"><td style="border:none;padding:0 14px 0 0;vertical-align:top"></td><td style="border:none;padding:0;vertical-align:top;text-align:left"><div style="font-size:12px;letter-spacing:1.5px;text-transform:uppercase;color:#0f6cbd;font-weight:700">Step 2</div><div style="font-size:20px;font-weight:600;margin:1px 0 6px 0">Choose the backup and where to restore it</div><div style="font-size:14px;line-height:1.55">Run the cell, then fill in the form that appears:<ul style="margin:6px 0 6px 0"><li><b>Backup container:</b> type the write-once container address and press <b>Browse</b>.</li><li><b>Backup folder</b> and <b>Backup (date and time):</b> pick from the lists. The newest complete backup is preselected.</li><li><b>Target workspace and lakehouse:</b> where the data will be restored.</li><li><b>Files, schemas, tables:</b> leave as they are to restore everything.</li><li><b>If the target is not empty:</b> leave on <i>Stop and change nothing</i> unless you have approval to delete from the target.</li></ul>Then press <b>Check inputs</b>. <b>Expect:</b> a line <i>Inputs accepted</i> listing the backup, the target and its IDs. A red line says what is missing or wrong. Changing anything afterwards means pressing <b>Check inputs</b> again.</div></td></tr></table></div>


In [ ]:
%run nb_restore_form

<div style="border-left:6px solid #8a6d00;border-radius:6px;padding:10px 16px;margin:14px 0 4px 0;background:rgba(127,127,127,0.09)"><table style="border:none;border-collapse:collapse;margin:0"><tr style="border:none;background:none"><td style="border:none;padding:0 14px 0 0;vertical-align:top"></td><td style="border:none;padding:0;vertical-align:top;text-align:left"><div style="font-size:12px;letter-spacing:1.5px;text-transform:uppercase;color:#8a6d00;font-weight:700">Step 3</div><div style="font-size:20px;font-weight:600;margin:1px 0 6px 0">Preview: see what the restore would do</div><div style="font-size:14px;line-height:1.55"><span style="display:inline-block;background:#107c10;color:#ffffff;border-radius:11px;padding:1px 10px;font-size:12px;font-weight:600">Changes nothing</span><br/>Reads the backup and the target and reports what would happen. Always run it, and read it, before step 4.<br/><b>Expect:</b> the backup and its source, the target, how many tables and Files items, and a <code>RESULT</code> line:<ul style="margin:6px 0 6px 0"><li><i>nothing is in the way</i>: go on to step 4.</li><li><i>the restore will STOP</i>: the items in the table below already exist in the target. Choose an empty lakehouse, or get approval to delete and change <i>If the target is not empty</i> on the form.</li><li><i>will be DELETED</i>: you chose to delete from the target. Check the list is what you expect before going on.</li></ul></div></td></tr></table></div>


In [ ]:
p = restore_form.params
if p is None:
    raise RuntimeError("Press Check inputs on the form first.")

clashes = preview_restore(
    p["archive"], p["backup_timestamp"], p["target_workspace_id"], p["target_lakehouse_id"],
    schemas=p["only_schemas"], tables=p["only_tables"],
    restore_files=p["restore_files"], clean_target=p["clean_target"],
)
if clashes:
    display(spark.createDataFrame(clashes))

<div style="border-left:6px solid #c4314b;border-radius:6px;padding:10px 16px;margin:14px 0 4px 0;background:rgba(127,127,127,0.09)"><table style="border:none;border-collapse:collapse;margin:0"><tr style="border:none;background:none"><td style="border:none;padding:0 14px 0 0;vertical-align:top"></td><td style="border:none;padding:0;vertical-align:top;text-align:left"><div style="font-size:12px;letter-spacing:1.5px;text-transform:uppercase;color:#c4314b;font-weight:700">Step 4</div><div style="font-size:20px;font-weight:600;margin:1px 0 6px 0">Restore</div><div style="font-size:14px;line-height:1.55"><span style="display:inline-block;background:#c4314b;color:#ffffff;border-radius:11px;padding:1px 10px;font-size:12px;font-weight:600">Writes to the target lakehouse</span> &nbsp; <span style="display:inline-block;background:#8a6d00;color:#ffffff;border-radius:11px;padding:1px 10px;font-size:12px;font-weight:600">Deletes first, only if you chose that</span><br/>Checks the backup against its manifest, then copies each table and, if ticked, everything under Files into the target. Files land directly under the target lakehouse&#39;s <code>Files</code> folder. A lakehouse of about 50 tables takes several minutes.<br/><b>Expect:</b> one line per table, ending with <code>RESTORE COMPLETE: &lt;timestamp&gt; | N of N tables restored | ...</code>, and a table with status <code>restored</code> on every row.<br/><b>If it ends with</b> <code>RESTORE FINISHED WITH PROBLEMS</code>: the table shows which rows failed and why. Tables already restored stay in place; fix the cause and restore the remaining ones by naming them on the form.<br/><b>Afterwards:</b> open the target lakehouse and spot-check a table or two. New tables can take a minute to appear in the SQL endpoint.</div></td></tr></table></div>


In [ ]:
p = restore_form.params
if p is None:
    raise RuntimeError("Press Check inputs on the form first.")

results = restore_from_archive(
    p["archive"], p["backup_timestamp"],
    tgt_ws_id=p["target_workspace_id"], tgt_lh_id=p["target_lakehouse_id"],
    schemas=p["only_schemas"], tables=p["only_tables"],
    restore_files=p["restore_files"],
    clean_target=p["clean_target"],
    confirm_target_name=p["confirm_target_name"],
)
display(spark.createDataFrame(results))

<div style="border-top:1px solid rgba(127,127,127,0.4);margin-top:18px;padding-top:8px;font-size:12px;opacity:0.8">Restore Utility &middot; backups are taken with the Backup Utility notebook. The archive is read-only here and is never modified by a restore.</div>